In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(81426, 31)


In [2]:
product_features = (
    df.groupby("skuNumber")
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        avg_qty=(
            "effective_qty",
            "mean"
        )
    )
    .reset_index()
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty
0,SKU00001,355,329,232,1.079027
1,SKU00009,686,608,356,1.128289
2,SKU00010,2152,1718,1186,1.252619
3,SKU00020,344,283,231,1.215548
4,SKU00022,1398,1349,993,1.036323


In [3]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category",
            "subCategory"
        ]
    ]
    .drop_duplicates()
)

product_features = (
    product_features.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory
0,SKU00001,355,329,232,1.079027,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum
1,SKU00009,686,608,356,1.128289,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum
2,SKU00010,2152,1718,1186,1.252619,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix
3,SKU00020,344,283,231,1.215548,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix
4,SKU00022,1398,1349,993,1.036323,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy


In [4]:
total_retailers = (
    df["customerId"]
    .nunique()
)

product_features[
    "popularity_score"
] = (
    product_features[
        "unique_buyers"
    ]
    / total_retailers
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score
0,SKU00001,355,329,232,1.079027,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.042797
1,SKU00009,686,608,356,1.128289,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.065671
2,SKU00010,2152,1718,1186,1.252619,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.218779
3,SKU00020,344,283,231,1.215548,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.042612
4,SKU00022,1398,1349,993,1.036323,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy,0.183177


In [5]:
(
    product_features[
        [
            "itemName",
            "popularity_score"
        ]
    ]
    .sort_values(
        "popularity_score",
        ascending=False
    )
    .head(10)
)

,itemName,popularity_score
9,Rajnigandha 4g,0.595462
14,TZ 00 (with Silver) 0.45g,0.591773
13,TZ 00 4.25g (6 Pcs),0.324110
10,RG Pearl Elaichi MP Pouch 0.14g,0.304372
189,Rajnigandha 17g Zipper 1 Pcs,0.241653
16,RG Pearl Elaichi 1.4g Hanger,0.225604
2,Pass Pass Meetha Magic 11.75g Hanger,0.218779
18,Pass Pass Meetha Magic 2.2g (100 pcs),0.201070
58,Center Fresh,0.185390
4,Pulse Kachha Aam 175 Candy,0.183177


In [6]:
product_features[
    "category_rank"
] = (
    product_features
    .groupby("category")
    ["total_qty"]
    .rank(
        ascending=False,
        method="dense"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score,category_rank
0,SKU00001,355,329,232,1.079027,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.042797,9.0
1,SKU00009,686,608,356,1.128289,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.065671,5.0
2,SKU00010,2152,1718,1186,1.252619,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.218779,3.0
3,SKU00020,344,283,231,1.215548,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.042612,8.0
4,SKU00022,1398,1349,993,1.036323,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy,0.183177,2.0


In [7]:
product_features.to_parquet(
    "../data/processed/product_features.parquet",
    index=False
)

print("Saved")

Saved
